In [ ]:
# ============================================================
# AI Innovators Lab: Fake News Detector
# ============================================================
# This notebook teaches the computer to look at a news headline.
# The goal is to predict whether the headline looks more like Fake news or Real news.
# A headline is the title of a news article.
# The computer will not truly "understand" news like a human.
# Instead, it learns word patterns from many examples.
# This project is for education only.
# It should not be used as a real fact-checking tool.

# Print a welcome message so students know the project has started.
print("Welcome to the Fake News Detector Project!")

# Print the main goal of the project in simple language.
print("Goal: Build an AI model that classifies news headlines as Fake or Real.")

# Print an important safety reminder about responsible AI use.
print("Important: This is for education only, not a real fact-checking system.")


In [ ]:
# ============================================================
# Step 1: Install and import required libraries
# ============================================================
# A library is a collection of helpful code written by other people.
# We import libraries so we do not have to build everything from scratch.

# Install the Kaggle library quietly using -q, so Colab does not print too much text.
!pip install -q kaggle

# Import os so Python can work with folders, file paths, and system settings.
import os

# Import zipfile so Python can open and extract zipped dataset files.
import zipfile

# Import shutil so Python can move, copy, and delete files or folders.
import shutil

# Import pandas and give it the short name pd.
# Pandas helps us work with tables, similar to spreadsheets.
import pandas as pd

# Import numpy and give it the short name np.
# NumPy helps us work with numbers, arrays, and mathematical operations.
import numpy as np

# Import matplotlib.pyplot and give it the short name plt.
# Matplotlib helps us create graphs and charts.
import matplotlib.pyplot as plt

# Import pickle so we can save the trained model to a file later.
import pickle

# Import files from google.colab so students can upload kaggle.json into Colab.
from google.colab import files

# Import train_test_split so we can divide data into training and testing parts.
from sklearn.model_selection import train_test_split

# Import TfidfVectorizer so we can convert text words into numbers.
from sklearn.feature_extraction.text import TfidfVectorizer

# Import LogisticRegression, the machine learning model used for this project.
from sklearn.linear_model import LogisticRegression

# Import accuracy_score to calculate how often the model is correct.
from sklearn.metrics import accuracy_score

# Import classification_report to show precision, recall, and F1-score.
from sklearn.metrics import classification_report

# Import confusion_matrix to see which predictions were correct or incorrect.
from sklearn.metrics import confusion_matrix

# Print a success message after all libraries are ready.
print("Libraries loaded successfully!")


In [ ]:
# ============================================================
# Step 2: Upload Kaggle API key
# ============================================================
# Kaggle stores many machine learning datasets.
# To download a Kaggle dataset directly in Colab, we need a small key file.
# That key file is named kaggle.json.
# Students should create it from their Kaggle account before running this cell.

# Open a file upload button in Google Colab.
uploaded = files.upload()

# Create the hidden Kaggle folder if it does not already exist.
# exist_ok=True means Python will not give an error if the folder is already there.
os.makedirs("/root/.kaggle", exist_ok=True)

# Check whether the uploaded files include kaggle.json.
if "kaggle.json" in uploaded:

    # Move kaggle.json into the folder where Kaggle expects to find it.
    shutil.move("kaggle.json", "/root/.kaggle/kaggle.json")

    # Change file permission to protect the API key.
    # 600 means only the owner can read and write the file.
    os.chmod("/root/.kaggle/kaggle.json", 600)

    # Tell students the upload worked.
    print("Kaggle API key uploaded successfully!")

# If kaggle.json was not uploaded, run this part instead.
else:

    # Tell students exactly what file they need to upload.
    print("Please upload your kaggle.json file before downloading the dataset.")


In [ ]:
# ============================================================
# Step 3: Download the dataset from Kaggle
# ============================================================
# A dataset is a collection of examples used to train an AI model.
# This dataset has fake news articles and real news articles.

# Use the Kaggle command to download the Fake and Real News Dataset.
!kaggle datasets download -d clmentbisaillon/fake-and-real-news-dataset

# Print a message after the download finishes.
print("Dataset downloaded!")


In [ ]:
# ============================================================
# Step 4: Unzip the dataset
# ============================================================
# Kaggle downloads the dataset as a compressed .zip file.
# We need to unzip it before pandas can read the CSV files inside.

# Store the name of the downloaded zip file in a variable.
zip_file = "fake-and-real-news-dataset.zip"

# Store the folder name where the unzipped files will be placed.
extract_folder = "fake_real_news_dataset"

# Check whether the folder already exists from an earlier run.
if os.path.exists(extract_folder):

    # Delete the old folder so we start with a clean copy of the dataset.
    shutil.rmtree(extract_folder)

# Open the zip file in read mode.
with zipfile.ZipFile(zip_file, "r") as zip_ref:

    # Extract all files from the zip file into the dataset folder.
    zip_ref.extractall(extract_folder)

# Print a message after the files are extracted.
print("Dataset extracted successfully!")


In [ ]:
# ============================================================
# Step 5: Load Fake.csv and True.csv
# ============================================================
# CSV files store table data.
# Fake.csv contains fake news articles.
# True.csv contains real news articles.

# Build the file path for Fake.csv.
fake_path = os.path.join(extract_folder, "Fake.csv")

# Build the file path for True.csv.
true_path = os.path.join(extract_folder, "True.csv")

# Read Fake.csv into a pandas table called fake_news.
fake_news = pd.read_csv(fake_path)

# Read True.csv into a pandas table called real_news.
real_news = pd.read_csv(true_path)

# Print a heading before showing fake news examples.
print("Fake news data:")

# Show the first five rows of the fake news table.
print(fake_news.head())

# Print a heading before showing real news examples.
print("\nReal news data:")

# Show the first five rows of the real news table.
print(real_news.head())

# Print how many fake articles are in the dataset.
print("\nNumber of fake articles:", len(fake_news))

# Print how many real articles are in the dataset.
print("Number of real articles:", len(real_news))


In [ ]:
# ============================================================
# Step 6: Create labels
# ============================================================
# A label is the correct answer for each example.
# In this project:
# label 0 means Fake.
# label 1 means Real.

# Add a new column named label to the fake news table.
# Every fake news row gets the value 0.
fake_news["label"] = 0

# Add a new column named label to the real news table.
# Every real news row gets the value 1.
real_news["label"] = 1

# Combine the fake news table and the real news table into one larger table.
# axis=0 means we stack rows on top of each other.
data = pd.concat([fake_news, real_news], axis=0)

# Shuffle the rows so fake and real examples are mixed together.
# frac=1 means use 100 percent of the rows.
# random_state=42 makes the shuffle repeatable.
# reset_index(drop=True) gives the shuffled table a clean new row number.
data = data.sample(frac=1, random_state=42).reset_index(drop=True)

# Print a message telling students the combined dataset is ready.
print("Combined dataset created!")

# Show the title and label columns for the first few rows.
print(data[["title", "label"]].head())


In [ ]:
# ============================================================
# Step 7: Use only headlines/titles
# ============================================================
# To keep the project beginner friendly, we use only the article title.
# The title is the headline that students will type later.
# Input to the model = headline.
# Output from the model = Fake or Real.

# Keep only the title column and the label column.
# dropna() removes rows where the title or label is missing.
data = data[["title", "label"]].dropna()

# Remove duplicate headlines so the same headline does not appear more than once.
data = data.drop_duplicates(subset=["title"])

# Print the final number of usable examples.
print("Final dataset size:", len(data))

# Show the first few rows of the cleaned dataset.
print(data.head())


In [ ]:
# ============================================================
# Step 8: Explore label counts
# ============================================================
# Before training a model, we should understand our data.
# Here, we count how many examples are Fake and how many are Real.

# Count how many times each label appears.
# sort_index() keeps label 0 before label 1.
label_counts = data["label"].value_counts().sort_index()

# Print a heading for the label counts.
print("\nLabel counts:")

# Print the number of fake examples.
# get(0, 0) means return 0 if label 0 is missing.
print("Fake:", label_counts.get(0, 0))

# Print the number of real examples.
# get(1, 0) means return 0 if label 1 is missing.
print("Real:", label_counts.get(1, 0))

# Create a new figure for the bar chart.
plt.figure(figsize=(6, 4))

# Create a bar chart showing fake and real counts.
plt.bar(["Fake", "Real"], [label_counts.get(0, 0), label_counts.get(1, 0)])

# Add a title to the chart.
plt.title("Number of Fake and Real News Headlines")

# Add a label to the horizontal axis.
plt.xlabel("News Type")

# Add a label to the vertical axis.
plt.ylabel("Count")

# Display the chart.
plt.show()


In [ ]:
# ============================================================
# Step 9: Split data into training and testing sets
# ============================================================
# Training data teaches the AI model.
# Testing data checks whether the AI can handle examples it has not seen before.

# Store the headlines in X.
# In machine learning, X usually means input features.
X = data["title"]

# Store the correct answers in y.
# In machine learning, y usually means labels or targets.
y = data["label"]

# Split the data into training and testing sets.
X_train, X_test, y_train, y_test = train_test_split(

    # These are the input headlines.
    X,

    # These are the correct labels.
    y,

    # Use 25 percent of the data for testing.
    test_size=0.25,

    # Use a fixed random state so results stay the same each time.
    random_state=42,

    # Keep the fake/real balance similar in both training and testing sets.
    stratify=y
)

# Print how many headlines will be used for training.
print("Training headlines:", len(X_train))

# Print how many headlines will be used for testing.
print("Testing headlines:", len(X_test))


In [ ]:
# ============================================================
# Step 10: Convert headlines into numbers
# ============================================================
# AI models cannot directly understand words like humans do.
# We must convert text into numbers first.
# TF-IDF gives higher importance to words that are useful for classification.
# Example idea:
# A common word like "the" is usually not very helpful.
# A more meaningful word pattern may help the model separate Fake from Real.

# Create a TF-IDF vectorizer.
# This object learns a vocabulary and converts headlines into number patterns.
vectorizer = TfidfVectorizer(

    # Convert all letters to lowercase so "News" and "news" are treated the same.
    lowercase=True,

    # Remove common English words such as "the", "and", and "is".
    stop_words="english",

    # Keep only the 5000 most useful word features to make the project faster.
    max_features=5000
)

# Learn the vocabulary from training headlines and convert them into numbers.
# fit_transform means "learn from this text, then transform it."
X_train_vectorized = vectorizer.fit_transform(X_train)

# Convert the testing headlines into numbers using the same learned vocabulary.
# We use transform, not fit_transform, because the test set should not teach the vectorizer.
X_test_vectorized = vectorizer.transform(X_test)

# Print a message saying the text has been converted.
print("Headlines converted into numbers!")

# Print how many word features the model can use.
print("Number of text features:", len(vectorizer.get_feature_names_out()))


In [ ]:
# ============================================================
# Step 11: Build and train the AI model
# ============================================================
# Logistic Regression is a beginner friendly classification model.
# It learns which word patterns are more connected with Fake or Real headlines.
# Even though the name says "Regression", here we use it for classification.

# Create the Logistic Regression model.
# max_iter=1000 gives the model enough attempts to finish learning.
model = LogisticRegression(max_iter=1000)

# Train the model using the vectorized training headlines and their correct labels.
# During training, the model learns word weights for Fake and Real.
model.fit(X_train_vectorized, y_train)

# Print a message when training is complete.
print("Model training completed!")


In [ ]:
# ============================================================
# Step 12: Evaluate the model
# ============================================================
# Evaluation means checking how well the model performs.
# We use the testing data because the model did not train on those examples.

# Ask the trained model to predict labels for the testing headlines.
y_pred = model.predict(X_test_vectorized)

# Compare the model predictions with the true answers.
# Accuracy means the fraction of predictions that were correct.
accuracy = accuracy_score(y_test, y_pred)

# Print the accuracy as a percentage.
print("Model Accuracy:", round(accuracy * 100, 2), "%")

# Print a heading for more detailed results.
print("\nClassification Report:")

# Print precision, recall, and F1-score for Fake and Real classes.
print(classification_report(

    # These are the true labels.
    y_test,

    # These are the predicted labels.
    y_pred,

    # These names make the report easier to read.
    target_names=["Fake", "Real"]
))

# Print a heading before the confusion matrix.
print("\nConfusion Matrix:")

# Print the confusion matrix as a table of numbers.
print(confusion_matrix(y_test, y_pred))


In [ ]:
# ============================================================
# Step 13: Visualize confusion matrix
# ============================================================
# A confusion matrix shows correct and incorrect predictions.
# Top-left     = Fake predicted as Fake.
# Top-right    = Fake predicted as Real.
# Bottom-left  = Real predicted as Fake.
# Bottom-right = Real predicted as Real.

# Create the confusion matrix using true labels and predicted labels.
cm = confusion_matrix(y_test, y_pred)

# Create a new figure for the confusion matrix image.
plt.figure(figsize=(5, 4))

# Display the confusion matrix as an image.
plt.imshow(cm)

# Add a chart title.
plt.title("Confusion Matrix")

# Add a color bar so larger numbers are easier to notice.
plt.colorbar()

# Label the x-axis tick marks as Fake and Real.
plt.xticks([0, 1], ["Fake", "Real"])

# Label the y-axis tick marks as Fake and Real.
plt.yticks([0, 1], ["Fake", "Real"])

# Label the horizontal axis as predicted labels.
plt.xlabel("Predicted Label")

# Label the vertical axis as true labels.
plt.ylabel("True Label")

# Loop through each row of the 2-by-2 confusion matrix.
for i in range(2):

    # Loop through each column of the 2-by-2 confusion matrix.
    for j in range(2):

        # Write the number inside the correct box of the matrix.
        plt.text(j, i, cm[i, j], ha="center", va="center")

# Display the final confusion matrix chart.
plt.show()


In [ ]:
# ============================================================
# Step 14: Test the model with example headlines
# ============================================================
# These are sample headlines for students to test.
# Students can change these headlines and run the cell again.
# Reminder: this model is educational, not a real fact-checker.

# Create a Python list of example headlines.
example_headlines = [

    # Example headline about science and space.
    "NASA announces new discovery about Mars exploration",

    # Example headline with suspicious dramatic wording.
    "Celebrity reveals secret cure that doctors do not want you to know",

    # Example headline about government and education.
    "Government officials announce new education funding plan",

    # Example headline about misinformation.
    "Scientists warn fake social media posts spread misinformation quickly",

    # Example headline about a local school event.
    "Breaking: Local school wins national robotics competition"
]

# Convert the example headlines into numbers using the same vectorizer.
example_vectorized = vectorizer.transform(example_headlines)

# Ask the trained model to predict Fake or Real for each example headline.
example_predictions = model.predict(example_vectorized)

# Ask the model for probability scores for each example.
# Probabilities help us estimate how confident the model is.
example_probabilities = model.predict_proba(example_vectorized)

# Print a heading before showing predictions.
print("Example Headline Predictions:\n")

# Loop through headline, prediction, and probability together.
for headline, prediction, probability in zip(example_headlines, example_predictions, example_probabilities):

    # Convert the numeric prediction into a readable word.
    # Prediction 1 means Real, and prediction 0 means Fake.
    label = "REAL" if prediction == 1 else "FAKE"

    # Choose the larger probability as the confidence score.
    confidence = max(probability) * 100

    # Print the original headline.
    print("Headline:", headline)

    # Print the model prediction.
    print("Prediction:", label)

    # Print the confidence score rounded to two decimal places.
    print("Confidence:", round(confidence, 2), "%")

    # Print a divider line to separate each example.
    print("-" * 70)


In [ ]:
# ============================================================
# Step 15: Let students type their own headline
# ============================================================
# This makes the project interactive.
# Students can type a headline and ask the model to classify it.
# Use made-up or public example headlines.
# Do not use this as a real news verification tool.

# Ask the student to type a news headline.
student_headline = input("Type a news headline to test: ")

# Put the student headline inside a list because the vectorizer expects a list of texts.
# Then convert the headline into numbers.
student_vectorized = vectorizer.transform([student_headline])

# Predict whether the student headline is Fake or Real.
# [0] gets the first prediction from the prediction list.
student_prediction = model.predict(student_vectorized)[0]

# Get the model probability scores for the student headline.
# [0] gets the first probability row.
student_probability = model.predict_proba(student_vectorized)[0]

# Check whether the model predicted label 1.
if student_prediction == 1:

    # If the prediction is 1, print Real.
    print("\nThe AI predicts: REAL")

# If the prediction is not 1, it must be 0 in this project.
else:

    # If the prediction is 0, print Fake.
    print("\nThe AI predicts: FAKE")

# Print the model confidence as a percentage.
print("Confidence:", round(max(student_probability) * 100, 2), "%")

# Print a responsible AI reminder.
print("\nReminder: This model is for education only. It is not a real fact-checking tool.")


In [ ]:
# ============================================================
# Step 16: Show words associated with Fake and Real headlines
# ============================================================
# This helps students understand what the model learned.
# Logistic Regression gives a weight to each word feature.
# Positive weights lean toward Real.
# Negative weights lean toward Fake.

# Get the vocabulary words learned by the TF-IDF vectorizer.
# Convert them into a NumPy array so we can index them easily.
feature_names = np.array(vectorizer.get_feature_names_out())

# Get the learned word weights from the Logistic Regression model.
# model.coef_[0] contains one weight for each word feature.
coefficients = model.coef_[0]

# Find the indices of the 20 largest positive weights.
# These are the words most strongly connected with Real headlines.
top_real_indices = np.argsort(coefficients)[-20:]

# Find the indices of the 20 smallest negative weights.
# These are the words most strongly connected with Fake headlines.
top_fake_indices = np.argsort(coefficients)[:20]

# Use the real-word indices to get the actual words.
top_real_words = feature_names[top_real_indices]

# Use the fake-word indices to get the actual words.
top_fake_words = feature_names[top_fake_indices]

# Print a heading for words associated with Real headlines.
print("\nWords more associated with REAL headlines:")

# Print the Real-associated words from strongest to weaker.
for word in top_real_words[::-1]:

    # Print one word on each line.
    print("-", word)

# Print a heading for words associated with Fake headlines.
print("\nWords more associated with FAKE headlines:")

# Print the Fake-associated words.
for word in top_fake_words:

    # Print one word on each line.
    print("-", word)

# Create a new figure for the Real word-weight chart.
plt.figure(figsize=(8, 6))

# Make a horizontal bar chart for words associated with Real headlines.
plt.barh(top_real_words, coefficients[top_real_indices])

# Add a title to the Real word chart.
plt.title("Words Associated with Real News")

# Label the horizontal axis as model weight.
plt.xlabel("Model Weight")

# Label the vertical axis as words.
plt.ylabel("Words")

# Show the Real word chart.
plt.show()

# Create a new figure for the Fake word-weight chart.
plt.figure(figsize=(8, 6))

# Make a horizontal bar chart for words associated with Fake headlines.
plt.barh(top_fake_words, coefficients[top_fake_indices])

# Add a title to the Fake word chart.
plt.title("Words Associated with Fake News")

# Label the horizontal axis as model weight.
plt.xlabel("Model Weight")

# Label the vertical axis as words.
plt.ylabel("Words")

# Show the Fake word chart.
plt.show()


In [ ]:
# ============================================================
# Step 17: Save the model and vectorizer
# ============================================================
# Saving allows us to reuse the trained model later.
# For text models, we must save two things:
# 1. The trained model.
# 2. The vectorizer that converts headlines into numbers.
# Without the vectorizer, the saved model would not know how to read text.

# Open a new file named fake_news_detector_model.pkl in write-binary mode.
with open("fake_news_detector_model.pkl", "wb") as file:

    # Save the trained Logistic Regression model into the file.
    pickle.dump(model, file)

# Open a new file named headline_vectorizer.pkl in write-binary mode.
with open("headline_vectorizer.pkl", "wb") as file:

    # Save the TF-IDF vectorizer into the file.
    pickle.dump(vectorizer, file)

# Print the name of the saved model file.
print("Model saved as fake_news_detector_model.pkl")

# Print the name of the saved vectorizer file.
print("Vectorizer saved as headline_vectorizer.pkl")

# ============================================================
# Step 18: Reflection questions for final showcase
# ============================================================
# These questions help students explain their project.
# They are useful for a final presentation or poster.

# Create a list of reflection questions.
reflection_questions = [

    # Ask students to explain the problem.
    "1. What problem does your AI model try to solve?",

    # Ask students to identify the dataset.
    "2. What dataset did you use?",

    # Ask students to explain the model input.
    "3. What input did the model use?",

    # Ask students to explain the model output.
    "4. What does the model predict?",

    # Ask students to report the accuracy.
    "5. What was your model accuracy?",

    # Ask students to think about model mistakes.
    "6. Did the model make any mistakes?",

    # Ask students to think about why this task is hard.
    "7. Why is fake news detection difficult?",

    # Ask students to think about responsible AI use.
    "8. Why should this model not be used as a real fact-checker?",

    # Ask students to suggest future improvements.
    "9. How could this AI system be improved?"
]

# Print a heading before the questions.
print("\nReflection Questions:")

# Loop through each reflection question.
for question in reflection_questions:

    # Print the current question.
    print(question)


In [ ]:
# ============================================================
# Presentation Template for Students
# ============================================================
# This template gives students a simple structure for presenting their project.
# Students can copy this text into slides or a poster.

# Store the full presentation template inside a multi-line string.
presentation_template = """
Project Title:
Fake News Detector AI Model

Team Members:
[Names]

Problem:
We wanted to build an AI model that predicts whether a news headline may be fake or real.

Dataset:
We used the Fake and Real News Dataset from Kaggle.
Dataset Link:
https://www.kaggle.com/datasets/clmentbisaillon/fake-and-real-news-dataset

Input:
News headline

Prediction:
Fake or Real

Method Used:
TF-IDF Vectorization and Logistic Regression

Result:
Our model reached approximately ____% accuracy.

Demo:
We typed a headline, and the model predicted whether it was fake or real.

What We Learned:
We learned how AI can classify text by learning patterns from labeled examples.

Responsible AI Note:
This project is for education only. It should not be used as a real fact-checking tool.

Future Improvement:
We could use full article text, verified fact-checking sources, newer data, and more advanced language models.
"""

# Print the presentation template so students can see and copy it.
print(presentation_template)
